# BizLens v2.2.12 - Decision Trees & Random Forests (Comprehensive)

Classification using **Decision Trees** and **Random Forests** with BizLens.
We will predict Titanic survival using the built-in Titanic dataset.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "bizlens", "seaborn"])
print("✅ BizLens v2.2.12 ready for Decision Trees & Random Forests!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sns.set_style("whitegrid")
# bl.set_profiling(True)

## 1. Load Titanic Dataset using BizLens

In [ ]:
df = bl.load_dataset("titanic")
print(f"Dataset shape: {df.shape}")
bl.describe(df)

## 2. Data Quality & Exploration

In [ ]:
bl.quality.data_profile(df)
bl.quality.completeness_report(df)

## 3. Data Preprocessing

In [ ]:
df_model = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']].copy()
df_model['age'] = df_model['age'].fillna(df_model['age'].median())
df_model = pd.get_dummies(df_model, columns=['sex', 'pclass'], drop_first=True)

X = df_model.drop('survived', axis=1)
y = df_model['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Data split completed.")

## 4. Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree Performance:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Visualize the Decision Tree
plt.figure(figsize=(16, 10))
plot_tree(dt, feature_names=X.columns, class_names=['Died', 'Survived'], filled=True, rounded=True)
plt.title("Decision Tree - Titanic Survival Prediction")
plt.show()

## 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Performance:")
print(classification_report(y_test, y_pred_rf))

## 6. Feature Importance

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importance.values, y=importance.index)
plt.title("Feature Importance - Random Forest")
plt.show()

## 7. ROC Curve Comparison

In [ ]:
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt.predict_proba(X_test)[:, 1])
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf.predict_proba(X_test)[:, 1])

plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1]):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]):.3f})')
plt.plot([0, 1], [0, 1], 'r--')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()